# The WSPD Heuristic based Repairer

Takes a NN tour and repairs all the wspd violations.

In [10]:
import tsplib95
from pathlib import Path
import numpy as np
from any_metric_wspd import gen_wspd
from wspd import build_wspd_np, point as wsp_point
from numba import njit

TSP_LIB_PATH = Path("TSPLIB") # ALL_tsp
DIMENSION_THRESHOLD = 4000
METRIC_THRESHOLD = min(4000, DIMENSION_THRESHOLD)
METRIC_CHECK = True
METRIC_FIX = False
ONLY_EUC2D = True
TOL=1e-2

In [11]:
def is_metric(dist_matrix: np.ndarray, tol=1e-9) -> bool:
    """Check if a distance matrix is metric (satisfies triangle inequality)."""
    if dist_matrix.ndim != 2 or dist_matrix.shape[0] != dist_matrix.shape[1]:
        raise ValueError("Input must be a square matrix.")
    # check non-negativity
    if not np.all(dist_matrix >= 0):
        return False
    # check zero diagonal
    if not np.allclose(np.diag(dist_matrix), 0.0, atol=tol):
        return False
    # check symmetry
    if not np.allclose(dist_matrix, dist_matrix.T, atol=tol):
        return False
    # check triangle inequality
    inequality_holds = (
        dist_matrix[:, np.newaxis, :]
        <= dist_matrix[:, :, np.newaxis] + dist_matrix[np.newaxis, :, :] + tol
    )
    return np.all(inequality_holds)  # pyright: ignore[reportReturnType]

In [12]:
all_problems : dict[str, tsplib95.models.StandardProblem] = {}

for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
    if file.suffix != ".tsp" or not file.is_file():
        continue
    problem = tsplib95.load(f"{file.absolute()}")
    if problem.dimension > DIMENSION_THRESHOLD or not problem.is_complete():
            continue
    if problem.edge_weight_type == "EXPLICIT":
        if METRIC_CHECK and not is_metric(dist_mat := problem._create_explicit_matrix().to_numpy(), tol=TOL):
            if METRIC_FIX and problem.is_full_matrix() and is_metric((dist_mat + dist_mat.T) / 2, tol=TOL):
                print(f"Fixing {problem.name} since it could be made metric by symmetrizing")
                problem.edge_weight_type = "EXPLICIT2"
            else:
                continue
        print(f"Adding {problem.name} since it's metric")
    
    if ONLY_EUC2D and problem.edge_weight_type != "EUC_2D":
        continue
    all_problems[problem.name] = problem
    print(f"Added {problem.name}")

print("Found", len(all_problems), "euclidean TSPs")

Added a280
Adding bayg29 since it's metric
Added berlin52
Added bier127
Added ch130
Added ch150
Added d1291
Added d1655
Added d198
Added d2103
Added d493
Added d657
Added eil101
Added eil51
Added eil76
Added fl1400
Added fl1577
Added fl3795
Added fl417
Added gil262
Added kroA100
Added kroA150
Added kroA200
Added kroB100
Added kroB150
Added kroB200
Added kroC100
Added kroD100
Added kroE100
Added lin105
Added lin318
Added lin318
Added nrw1379
Added p654
Added pcb1173
Added pcb3038
Added pcb442
Added pr1002
Added pr107
Added pr124
Added pr136
Added pr144
Added pr152
Added pr226
Added pr2392
Added pr264
Added pr299
Added pr439
Added pr76
Added rat195
Added rat575
Added rat783
Added rat99
Added rd100
Added rd400
Added rl1304
Added rl1323
Added rl1889
Adding si1032 since it's metric
Adding si175 since it's metric
Adding si535 since it's metric
Added st70
Added ts225
Added tsp225
Added u1060
Added u1432
Added u159
Added u1817
Added u2152
Added u2319
Added u574
Added u724
Added vm1084
Added vm

In [13]:
def geo_based_dist_matrix(nodes_array, radius: float = 6378.388) -> np.ndarray:
    lat = np.deg2rad(nodes_array[:, 0])
    lon = np.deg2rad(nodes_array[:, 1])

    # Broadcast to pairwise differences
    lat_i = lat[:, None]
    lat_j = lat[None, :]
    lon_i = lon[:, None]
    lon_j = lon[None, :]

    q1 = np.cos(lon_i - lon_j)
    q2 = np.cos(lat_i - lat_j)
    q3 = np.cos(lat_i + lat_j)

    return radius * np.arccos(0.5 * ((1 + q1) * q2 - (1 - q1) * q3))

def convert_to_dist_matrix(problem: tsplib95.models.StandardProblem) -> np.ndarray:
    """Convert a tsplib95 StandardProblem to a distance matrix."""
    n = int(problem.dimension)  # Cast to int to resolve type mismatch
    dist_matrix = np.zeros((n, n))
    nodes_array = np.array(list(problem.node_coords.values()))
    if problem.edge_weight_type in ["EUC_2D", "CEIL_2D", "ATT"]:
        dist_matrix = np.linalg.norm(nodes_array[:, np.newaxis] - nodes_array[np.newaxis, :], axis=2)
        if problem.edge_weight_type == "ATT":
            dist_matrix /= np.sqrt(10)
    elif "GEO" in problem.edge_weight_type:
        dist_matrix = geo_based_dist_matrix(nodes_array)
    elif "EXPLICIT" in problem.edge_weight_type:
        dist_matrix = problem._create_explicit_matrix().to_numpy()
    elif "EXPLICIT2" in problem.edge_weight_type:
        dist_matrix = (dist_matrix + dist_matrix.T) / 2
    else:
        raise ValueError(f"Unsupported edge weight type: {problem.edge_weight_type}")
    if "CEIL" in problem.edge_weight_type:
        dist_matrix = np.ceil(dist_matrix)
    return dist_matrix


#all_dist_matrices = ((problem.name, convert_to_dist_matrix(problem)) for problem in all_problems.values() if int(problem.dimension) <= DIMENSION_THRESHOLD)

In [14]:
def nearest_neighbor_tour(dist_matrix: np.ndarray, start: int = 0) -> tuple[np.ndarray, float]:
    n = dist_matrix.shape[0]
    tour = np.empty(n, dtype=np.int32)
    visited = np.zeros(n, dtype=bool)

    tour[0] = start
    visited[start] = True

    for i in range(1, n):
        current = tour[i - 1]
        next_city = np.argmin(np.where(visited, np.inf, dist_matrix[current]))
        tour[i] = next_city
        visited[next_city] = True

    length = float(dist_matrix[tour, np.roll(tour, -1)].sum())
    return tour, length


nn_tours: dict[str, np.ndarray] = {}
nn_lengths: dict[str, float] = {}
dist_matrices: dict[str, np.ndarray] = {}

for name, prob in all_problems.items():
    D = convert_to_dist_matrix(prob)
    dist_matrices[name] = D
    tour, length = nearest_neighbor_tour(D, start=0)
    nn_tours[name] = tour
    nn_lengths[name] = length

print(f"Built NN tours for {len(nn_tours)} problems.")
print("Example:", next(iter(nn_lengths.items())))

Built NN tours for 69 problems.
Example: ('a280', 3148.1099349344026)


In [15]:
@njit(cache=True)
def wsp_heuristic_good(
    tour: np.ndarray,
    pos_in_tour: np.ndarray,
    A: np.ndarray,
    B: np.ndarray,
    state_arr: np.ndarray,
    S: float
) -> bool:
    """Checks WSP heuristic on a tour with at least 2 A<->B connections.
    Supports both non-wrapping tours [..] and wrapping tours [.., start]."""
    assert len(A) > 0 and len(B) > 0, "Sets must be non-empty"

    exit_A, exit_B, biconn_count = 0, 0, 0
    n = len(pos_in_tour)

    # Detect whether tour is explicitly closed (length n+1 and last == first)
    is_wrapped = False
    if tour.shape[0] == n + 1 and tour[n] == tour[0]:
        is_wrapped = True

    state_arr[A] = 1
    state_arr[B] = 2

    for node in A:
        pos = pos_in_tour[node]

        if is_wrapped:
            right_node = tour[pos + 1]
        else:
            right_idx = 0 if pos == n - 1 else pos + 1
            right_node = tour[right_idx]

        v = state_arr[right_node]
        if v == 0:
            exit_A += 1
        elif v == 2:
            biconn_count += 1

        left_idx = n - 1 if pos == 0 else pos - 1
        u = state_arr[tour[left_idx]]
        if u == 0:
            exit_A += 1

    for node in B:
        pos = pos_in_tour[node]

        if is_wrapped:
            right_node = tour[pos + 1]
        else:
            right_idx = 0 if pos == n - 1 else pos + 1
            right_node = tour[right_idx]

        v = state_arr[right_node]
        if v == 0:
            exit_B += 1
        elif v == 1:
            biconn_count += 1

        left_idx = n - 1 if pos == 0 else pos - 1
        u = state_arr[tour[left_idx]]
        if u == 0:
            exit_B += 1

    state_arr[A] = 0
    state_arr[B] = 0

    if exit_A == 0 or exit_B == 0:
        return biconn_count == 2
    elif exit_A % 2 == 0:
        if S >= 1.5:
            return biconn_count == 0
        return True
    else:
        return biconn_count == 1

In [16]:
def repair_tour(dist_matrix: np.ndarray, tour: np.ndarray, coords=None) -> np.ndarray:
    """Given a TSP and a non-wrapping tour, attempts to repair the tour by fixing tours that violate the WSP heuristic."""

    n = tour.shape[0]
    t = tour.astype(np.int32, copy=True)

    # Precompute WSPDs once
    if ONLY_EUC2D:
        points = [wsp_point(coord) for coord in coords]
        wspd_1 = [
            (np.asarray(A, dtype=np.int32), np.asarray(B, dtype=np.int32))
            for A, B in build_wspd_np(len(points), 2, 1.0*2, points)
            if len(A) > 1 and len(B) > 1
        ]
        wspd_15 = [
            (np.asarray(A, dtype=np.int32), np.asarray(B, dtype=np.int32))
            for A, B in build_wspd_np(len(points), 2, 1.0*1.5, points)
            if len(A) > 1 and len(B) > 1
        ]
    else:
        wspd_1 = [
            (np.asarray(A, dtype=np.int32), np.asarray(B, dtype=np.int32))
            for A, B in gen_wspd(dist_matrix, eps=0.99999)
            if len(A) > 1 and len(B) > 1
        ]
        wspd_15 = [
            (np.asarray(A, dtype=np.int32), np.asarray(B, dtype=np.int32))
            for A, B in gen_wspd(dist_matrix, eps=0.66666)
            if len(A) > 1 and len(B) > 1
        ]

    def build_closed_and_pos(curr_tour: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        closed = np.empty(n + 1, dtype=np.int32)
        closed[:-1] = curr_tour
        closed[-1] = curr_tour[0]
        pos = np.empty(n, dtype=np.int32)
        pos[curr_tour] = np.arange(n, dtype=np.int32)
        return closed, pos

    def bad_pairs(curr_tour: np.ndarray):
        closed, pos = build_closed_and_pos(curr_tour)
        state = np.zeros(n, dtype=np.uint8)
        bad = []

        for A, B in wspd_1:
            if not wsp_heuristic_good(closed, pos, A, B, state, 1.0):
                bad.append((A, B, 1.0))
        for A, B in wspd_15:
            if not wsp_heuristic_good(closed, pos, A, B, state, 1.5):
                bad.append((A, B, 1.5))
        return bad

    def try_two_opt_fix(curr_tour: np.ndarray, A: np.ndarray, B: np.ndarray) -> bool:
        pos = np.empty(n, dtype=np.int32)
        pos[curr_tour] = np.arange(n, dtype=np.int32)

        # Helper to find nodes that actually have edges leaving the set
        def get_exit_edges(nodes, pos_array):
            node_set = set(nodes)
            exit_edges = []
            for node in nodes:
                p = int(pos_array[node])
                # Check left edge
                if curr_tour[(p - 1) % n] not in node_set:
                    exit_edges.append((p - 1) % n) # The edge starts at p-1
                # Check right edge
                if curr_tour[(p + 1) % n] not in node_set:
                    exit_edges.append(p) # The edge starts at p
            return list(set(exit_edges)) # Deduplicate

        edges_A = get_exit_edges(A, pos)
        edges_B = get_exit_edges(B, pos)

        # Allow lateral moves or tiny regressions to fix topology (e.g., 1e-4)
        best_delta = 1e-4 
        best_move = None

        for u in edges_A:
            for v in edges_B:
                if u == v:
                    continue
                
                # Ensure u < v for clean slice reversal
                u_idx, v_idx = (u, v) if u < v else (v, u)
                u1 = (u_idx + 1) % n
                v1 = (v_idx + 1) % n

                if u1 == v_idx or (u_idx == 0 and v_idx == n - 1):
                    continue

                old_cost = dist_matrix[curr_tour[u_idx], curr_tour[u1]] + dist_matrix[curr_tour[v_idx], curr_tour[v1]]
                new_cost = dist_matrix[curr_tour[u_idx], curr_tour[v_idx]] + dist_matrix[curr_tour[u1], curr_tour[v1]]
                delta = new_cost - old_cost

                if delta < best_delta:
                    best_delta = delta
                    best_move = (u_idx, v_idx)

        if best_move is None or best_delta == 1e-4:
            return False

        u, v = best_move
        curr_tour[u + 1 : v + 1] = curr_tour[u + 1 : v + 1][::-1]
        return True

    max_iters = max(10, 3 * n)
    for _ in range(max_iters):
        violations = bad_pairs(t)
        if not violations:
            break

        changed = False
        for A, B, _ in violations:
            if try_two_opt_fix(t, A, B):
                changed = True
                break

        if not changed:
            break

    return t

In [17]:
for name, tour in nn_tours.items():
    D = dist_matrices[name]
    if ONLY_EUC2D:
        coords = np.array(list(all_problems[name].node_coords.values()))
        repaired = repair_tour(D, tour, coords=coords)
    else:
        repaired = repair_tour(D, tour)
    new_length = D[repaired, np.roll(repaired, -1)].sum()
    savings = (new_length / nn_lengths[name]) - 1.0
    if savings != 0.0:
        print(f"{name}: savings {savings:.2%}")
    #print(f"{name}: NN length {nn_lengths[name]:.2f}, repaired length {D[repaired, np.roll(repaired, -1)].sum():.2f}")

d1291: savings -0.05%
eil76: savings -1.94%
kroA200: savings -0.57%
p654: savings -2.16%
pcb3038: savings -0.01%
pr264: savings -1.89%
u1432: savings -0.04%
